In [1]:
import numpy as np 
import matplotlib.pyplot as plt 
import glob
import os
import pandas as pd
from turbo_seti.find_event.find_event_pipeline import find_event_pipeline
from turbo_seti.find_event.plot_event_pipeline import plot_event_pipeline
import multiprocessing
import logging

In [2]:
df = pd.read_csv('/datax/scratch/benjb/bl_nearby_stars/BL_cadences_unique_nearby_star_sample_only.csv')
df.iloc[6612:6614]['.dat path 1'].values

array(['/datax/scratch/benjb/bl_nearby_stars/seticore_output/beyond_5.1_pc_accelerated/blc51_guppi_58369_25405_HIP116971_0009.gpuspec.0000.dat',
       '/datax/scratch/benjb/bl_nearby_stars/seticore_output/beyond_5.1_pc_accelerated/blc52_guppi_58369_25405_HIP116971_0009.gpuspec.0000.dat'],
      dtype=object)

In [3]:
df.keys()

Index(['Target', 'RA', 'DEC', 'Distance (pc)', 'Cadence ID', 'Cadence Type',
       'Receiver', 'Vmag', 'Spectral Type', '.h5 path 1', '.h5 path 2',
       '.h5 path 3', '.h5 path 4', '.h5 path 5', '.h5 path 6', '.dat path 1',
       '.dat path 2', '.dat path 3', '.dat path 4', '.dat path 5',
       '.dat path 6'],
      dtype='object')

In [ ]:
paths = df['.h5 path 1'].values
path_bases = np.array([os.path.basename(path) for path in paths])
print(len(np.unique(paths)))
print(len(np.unique(path_bases)))

outdir = '/datax/scratch/benjb/bl_nearby_stars/seticore_output/dats_for_basename_ambiguity/'
relevant_h5_list = []

need_to_be_searched_again = []
for i in range(6):
    paths = df[f'.h5 path {i+1}'].values
    path_bases = np.array([os.path.basename(path) for path in paths])
    for j in range(len(df)):
        h5_path = paths[j]
        relevant_h5s = paths[np.where(path_bases == os.path.basename(h5_path))[0]]
        if (len(np.unique(relevant_h5s)) > 1):
            # search both these .h5 files again
            #print(relevant_h5s)
            for h5 in relevant_h5s:
                relevant_h5_list.append(h5)
                num = np.unique(np.where(paths == h5)[0])[0]
                print(num, h5)
                console = 'CUDA_DEVICE_ORDER=PCI_BUS_ID CUDA_VISIBLE_DEVICES=3 seticore ' + h5 + ' -M 4 -s 10 --output ' + outdir + str(num) + '_' + os.path.basename(h5)[:-2] + 'dat'
                os.system(console)

In [4]:
index_for_reassignment = []
ambfiles = glob.glob('/datax/scratch/benjb/bl_nearby_stars/seticore_output/dats_for_basename_ambiguity/*.dat')
for f in ambfiles:
    idx = os.path.basename(f).split('_')[0]
    index_for_reassignment.append(int(idx))
index_for_reassignment = np.unique(index_for_reassignment)
print(index_for_reassignment)
print(len(index_for_reassignment))

[ 6612  6613  6614  6615  6616  6635  6637  6638  6639  6640  6710  6711
  6712  6713  6714  6733  6735  6736  6737  6738  6796  6797  6798  6799
  6800  6819  6821  6822  6823  6824  6923  6924  6925  6926  6927  6946
  6948  6949  6950  6951  7014  7015  7016  7017  7018  7037  7039  7040
  7041  7042  9690  9691  9692  9693  9694  9713  9715  9716  9717  9718
  9751  9752  9753  9754  9755  9774  9776  9777  9778  9779  9813  9814
  9815  9816  9817  9836  9838  9839  9840  9841  9898  9899  9900  9901
  9902  9921  9923  9924  9925  9926  9945  9946  9947  9948  9949  9968
  9970  9971  9972  9973 15940 15941 15942 15943 15944 15963 15965 15966
 15967 15968 16058 16059 16060 16061 16062 16081 16083 16084 16085 16086
 16196 16197 16198 16199 16200 16219 16221 16222 16223 16224 16396 16397
 16417 16419 16549 16550 16571 16573]
138


In [5]:
dat_paths_1 = df['.dat path 1'].values 
dat_paths_2 = df['.dat path 2'].values 
dat_paths_3 = df['.dat path 3'].values 
dat_paths_4 = df['.dat path 4'].values 
dat_paths_5 = df['.dat path 5'].values 
dat_paths_6 = df['.dat path 6'].values 

ndp1 = []
ndp2 = []
ndp3 = []
ndp4 = []
ndp5 = []
ndp6 = []

ambdir = '/datax/scratch/benjb/bl_nearby_stars/seticore_output/dats_for_basename_ambiguity/'

for i in range(len(dat_paths_1)):
    d1 = dat_paths_1[i]
    d2 = dat_paths_2[i]
    d3 = dat_paths_3[i]
    d4 = dat_paths_4[i]
    d5 = dat_paths_5[i]
    d6 = dat_paths_6[i]
    if i in index_for_reassignment:
        ndp1.append(ambdir+str(i)+'_'+os.path.basename(d1))
        ndp2.append(ambdir+str(i)+'_'+os.path.basename(d2))
        ndp3.append(ambdir+str(i)+'_'+os.path.basename(d3))
        ndp4.append(ambdir+str(i)+'_'+os.path.basename(d4))
        ndp5.append(ambdir+str(i)+'_'+os.path.basename(d5))
        ndp6.append(ambdir+str(i)+'_'+os.path.basename(d6))
    else:
        ndp1.append(d1)
        ndp2.append(d2)
        ndp3.append(d3)
        ndp4.append(d4)
        ndp5.append(d5)
        ndp6.append(d6)

In [6]:
df.insert(15, 'turboSETI .dat path 6', ndp6)
df.insert(15, 'turboSETI .dat path 5', ndp5)
df.insert(15, 'turboSETI .dat path 4', ndp4)
df.insert(15, 'turboSETI .dat path 3', ndp3)
df.insert(15, 'turboSETI .dat path 2', ndp2)
df.insert(15, 'turboSETI .dat path 1', ndp1)
df

,Target,RA,DEC,Distance (pc),Cadence ID,Cadence Type,Receiver,Vmag,Spectral Type,.h5 path 1,...,turboSETI .dat path 3,turboSETI .dat path 4,turboSETI .dat path 5,turboSETI .dat path 6,.dat path 1,.dat path 2,.dat path 3,.dat path 4,.dat path 5,.dat path 6
0,GJ1002,0 6 42.9,-7 32 51,4.69,3635,ABACAD,Rcvr1_2,13.7,M5.0V,/datag/pipeline/AGBT16A_999_219/holding/splice...,...,/datax/scratch/benjb/bl_nearby_stars/seticore_...,/datax/scratch/benjb/bl_nearby_stars/seticore_...,/datax/scratch/benjb/bl_nearby_stars/seticore_...,/datax/scratch/benjb/bl_nearby_stars/seticore_...,/datax/scratch/benjb/bl_nearby_stars/seticore_...,/datax/scratch/benjb/bl_nearby_stars/seticore_...,/datax/scratch/benjb/bl_nearby_stars/seticore_...,/datax/scratch/benjb/bl_nearby_stars/seticore_...,/datax/scratch/benjb/bl_nearby_stars/seticore_...,/datax/scratch/benjb/bl_nearby_stars/seticore_...
1,GJ1002,0 6 42.9,-7 32 51,4.69,9642,ABACAD,Rcvr2_3,13.7,M5.0V,/datag/pipeline/AGBT16B_999_18/holding/spliced...,...,/datax/scratch/benjb/bl_nearby_stars/seticore_...,/datax/scratch/benjb/bl_nearby_stars/seticore_...,/datax/scratch/benjb/bl_nearby_stars/seticore_...,/datax/scratch/benjb/bl_nearby_stars/seticore_...,/datax/scratch/benjb/bl_nearby_stars/seticore_...,/datax/scratch/benjb/bl_nearby_stars/seticore_...,/datax/scratch/benjb/bl_nearby_stars/seticore_...,/datax/scratch/benjb/bl_nearby_stars/seticore_...,/datax/scratch/benjb/bl_nearby_stars/seticore_...,/datax/scratch/benjb/bl_nearby_stars/seticore_...
2,GJ1002,0 6 42.9,-7 32 51,4.69,12847,ABACAD,Rcvr4_6,13.7,M5.0V,/datag/pipeline/AGBT16B_999_82/collate/spliced...,...,/datax/scratch/benjb/bl_nearby_stars/seticore_...,/datax/scratch/benjb/bl_nearby_stars/seticore_...,/datax/scratch/benjb/bl_nearby_stars/seticore_...,/datax/scratch/benjb/bl_nearby_stars/seticore_...,/datax/scratch/benjb/bl_nearby_stars/seticore_...,/datax/scratch/benjb/bl_nearby_stars/seticore_...,/datax/scratch/benjb/bl_nearby_stars/seticore_...,/datax/scratch/benjb/bl_nearby_stars/seticore_...,/datax/scratch/benjb/bl_nearby_stars/seticore_...,/datax/scratch/benjb/bl_nearby_stars/seticore_...
3,GJ1002,0 6 42.9,-7 32 51,4.69,423617,ABACAD,Rcvr4_6,13.7,M5.0V,/datag/pipeline/AGBT23B_999_31/blc01_blp01/blc...,...,/datax/scratch/benjb/bl_nearby_stars/seticore_...,/datax/scratch/benjb/bl_nearby_stars/seticore_...,/datax/scratch/benjb/bl_nearby_stars/seticore_...,/datax/scratch/benjb/bl_nearby_stars/seticore_...,/datax/scratch/benjb/bl_nearby_stars/seticore_...,/datax/scratch/benjb/bl_nearby_stars/seticore_...,/datax/scratch/benjb/bl_nearby_stars/seticore_...,/datax/scratch/benjb/bl_nearby_stars/seticore_...,/datax/scratch/benjb/bl_nearby_stars/seticore_...,/datax/scratch/benjb/bl_nearby_stars/seticore_...
4,GJ1002,0 6 42.9,-7 32 51,4.69,423617,ABACAD,Rcvr4_6,13.7,M5.0V,/datag/pipeline/AGBT23B_999_31/blc02_blp02/blc...,...,/datax/scratch/benjb/bl_nearby_stars/seticore_...,/datax/scratch/benjb/bl_nearby_stars/seticore_...,/datax/scratch/benjb/bl_nearby_stars/seticore_...,/datax/scratch/benjb/bl_nearby_stars/seticore_...,/datax/scratch/benjb/bl_nearby_stars/seticore_...,/datax/scratch/benjb/bl_nearby_stars/seticore_...,/datax/scratch/benjb/bl_nearby_stars/seticore_...,/datax/scratch/benjb/bl_nearby_stars/seticore_...,/datax/scratch/benjb/bl_nearby_stars/seticore_...,/datax/scratch/benjb/bl_nearby_stars/seticore_...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
39172,SO0253,2 53 4.68,16 51 52,3.85,421558,ABACAD,Rcvr8_10_NEW,15.1,M6.5V,/datag/pipeline/AGBT23B_999_14/blc72_blp32/blc...,...,/datax/scratch/benjb/bl_nearby_stars/seticore_...,/datax/scratch/benjb/bl_nearby_stars/seticore_...,/datax/scratch/benjb/bl_nearby_stars/seticore_...,/datax/scratch/benjb/bl_nearby_stars/seticore_...,/datax/scratch/benjb/bl_nearby_stars/seticore_...,/datax/scratch/benjb/bl_nearby_stars/seticore_...,/datax/scratch/benjb/bl_nearby_stars/seticore_...,/datax/scratch/benjb/bl_nearby_stars/seticore_...,/datax/scratch/benjb/bl_nearby_stars/

In [7]:
df.iloc[6610:6614]['turboSETI .dat path 1'].values

array(['/datax/scratch/benjb/bl_nearby_stars/seticore_output/beyond_5.1_pc_accelerated/spliced_blc0001020304050607_guppi_57659_14619_HIP116971_0021.gpuspec.0000.dat',
       '/datax/scratch/benjb/bl_nearby_stars/seticore_output/beyond_5.1_pc_accelerated/spliced_blc00010203040506o7o0111213141516o7o0212223242526o7o031323334353637_guppi_58362_17007_HIP116971_0105.gpuspec.0000.dat',
       '/datax/scratch/benjb/bl_nearby_stars/seticore_output/dats_for_basename_ambiguity/6612_blc51_guppi_58369_25405_HIP116971_0009.gpuspec.0000.dat',
       '/datax/scratch/benjb/bl_nearby_stars/seticore_output/dats_for_basename_ambiguity/6613_blc52_guppi_58369_25405_HIP116971_0009.gpuspec.0000.dat'],
      dtype=object)

In [8]:
df.to_csv('/datax/scratch/benjb/bl_nearby_stars/BL_NSS_cadences_turboSETI.csv')